# WormProfiler crop model training

This Colab notebook trains a small TensorFlow/Keras segmentation model to identify the worm area from paired images and ilastik or Paint-style label masks.

Expected naming:

```text
images/IMAGE_1.png
labels/IMAGE_1_Labels.tiff
```

The trained model predicts a binary worm mask. WormProfiler can then find the largest predicted worm region, add a margin, and crop the original image around it.

## 1. Install and import dependencies

In [ ]:
!pip -q install imageio pillow imagecodecs tifffile

from pathlib import Path
import json
import math
import os
import random
import re

import cv2
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Set your folders

If your data is in Google Drive, uncomment the mount cell line and set the paths below.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Change these two paths.
IMAGE_DIR = Path("/content/drive/MyDrive/WormProfiler/images")
LABEL_DIR = Path("/content/drive/MyDrive/WormProfiler/labels")

# Where trained model files and plots will be written.
OUTPUT_DIR = Path("/content/drive/MyDrive/WormProfiler/model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training settings.
IMG_SIZE = (256, 256)  # height, width. Keep this fixed for export/deployment.
BATCH_SIZE = 8
EPOCHS = 60
VALIDATION_SPLIT = 0.20
TEST_SPLIT = 0.10
SEED = 42

# Label/mask settings.
# LABEL_MASK_MODE options:
# - "auto": grayscale/indexed labels use WORM_LABEL_VALUE; RGB/RGBA labels use the painted color.
# - "value": worm pixels are exactly WORM_LABEL_VALUE in an indexed/grayscale label image.
# - "paint_color": worm pixels are a solid color painted in Paint or another editor.
# - "non_background": worm pixels are anything that differs from the background color.
LABEL_MASK_MODE = "auto"

# Used by LABEL_MASK_MODE="value". If None, use the highest non-zero label value.
# Examples: 1 for masks with values [0, 1], 2 for ilastik labels [0, 1, 2], 255 for binary masks.
WORM_LABEL_VALUE = None

# Used by Paint-style RGB/RGBA masks. Leave as None to infer from each label image.
# Examples: PAINT_WORM_RGB = (255, 0, 0), PAINT_BACKGROUND_RGB = (0, 0, 0)
PAINT_WORM_RGB = None
PAINT_BACKGROUND_RGB = None
PAINT_COLOR_TOLERANCE = 35
PAINT_ALPHA_THRESHOLD = 10

# Prediction/crop settings for preview only. WormProfiler can use the same ideas.
PREDICTION_THRESHOLD = 0.50
CROP_MARGIN_PIXELS_ON_RESIZED_IMAGE = 20

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Match images to label masks

This matches `IMAGE_1.png` to `IMAGE_1_Labels.tiff` by removing the `_Labels` suffix from the label filename.

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
LABEL_EXTS = {".png", ".bmp", ".jpg", ".jpeg", ".tif", ".tiff"}

def normalized_stem(path):
    return path.stem.lower()

def label_key(path):
    stem = normalized_stem(path)
    return re.sub(r"_labels$", "", stem)

image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS])
label_files = sorted([p for p in LABEL_DIR.iterdir() if p.suffix.lower() in LABEL_EXTS])

images_by_key = {normalized_stem(p): p for p in image_files}
labels_by_key = {label_key(p): p for p in label_files}

pairs = []
missing_images = []
for key, label_path in labels_by_key.items():
    image_path = images_by_key.get(key)
    if image_path is None:
        missing_images.append(label_path.name)
        continue
    pairs.append((image_path, label_path))

missing_labels = [p.name for key, p in images_by_key.items() if key not in labels_by_key]

print(f"Images found: {len(image_files)}")
print(f"Labels found: {len(label_files)}")
print(f"Matched pairs: {len(pairs)}")

if missing_images[:10]:
    print("Labels with no matching image, first 10:", missing_images[:10])
if missing_labels[:10]:
    print("Images with no matching label, first 10:", missing_labels[:10])

if not pairs:
    raise RuntimeError("No image/label pairs found. Check IMAGE_DIR, LABEL_DIR, and filename suffixes.")

pairs[:5]

## 4. Inspect label values and choose the worm class

Ilastik label images can look black because class IDs are small values such as `0`, `1`, or `2`. Paint-style masks are often RGB images, so this cell also prints the inferred painted foreground/background colors and mask coverage. If the automatically chosen mode or color is wrong, set `LABEL_MASK_MODE`, `WORM_LABEL_VALUE`, `PAINT_WORM_RGB`, or `PAINT_BACKGROUND_RGB` manually above and rerun from here.

In [ ]:
def read_label_image(path):
    return iio.imread(path)

def read_label_array(path):
    arr = read_label_image(path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr

def _rgb_like(arr):
    return arr.ndim == 3 and arr.shape[-1] >= 3

def _rgb_view(arr):
    rgb = arr[..., :3]
    if np.issubdtype(rgb.dtype, np.floating):
        scale = 255.0 if float(np.nanmax(rgb)) <= 1.0 else 1.0
        rgb = np.clip(rgb * scale, 0, 255)
    elif rgb.dtype != np.uint8:
        info = np.iinfo(rgb.dtype) if np.issubdtype(rgb.dtype, np.integer) else None
        if info is not None and info.max > 255:
            rgb = rgb.astype(np.float32) * (255.0 / info.max)
        rgb = np.clip(rgb, 0, 255)
    return rgb.astype(np.uint8)

def _label_mode_for_array(arr):
    if LABEL_MASK_MODE != "auto":
        return LABEL_MASK_MODE
    if not _rgb_like(arr):
        return "value"
    rgb = _rgb_view(arr)
    if np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 0], rgb[..., 2]):
        return "value"
    return "paint_color"

def _color_distance(rgb, target_rgb):
    target = np.asarray(target_rgb, dtype=np.float32)
    return np.sqrt(np.sum((rgb.astype(np.float32) - target) ** 2, axis=-1))

def _edge_pixels(rgb):
    return np.concatenate([rgb[0, :, :], rgb[-1, :, :], rgb[:, 0, :], rgb[:, -1, :]], axis=0)

def _most_common_color(pixels):
    colors, counts = np.unique(pixels.reshape(-1, 3), axis=0, return_counts=True)
    return colors[int(np.argmax(counts))]

def _estimate_background_rgb(rgb):
    if PAINT_BACKGROUND_RGB is not None:
        return np.asarray(PAINT_BACKGROUND_RGB, dtype=np.float32)
    return _most_common_color(_edge_pixels(rgb)).astype(np.float32)

def _estimate_worm_rgb(rgb):
    if PAINT_WORM_RGB is not None:
        return np.asarray(PAINT_WORM_RGB, dtype=np.float32)

    background_rgb = _estimate_background_rgb(rgb)
    candidate_mask = _color_distance(rgb, background_rgb) > PAINT_COLOR_TOLERANCE
    if not np.any(candidate_mask):
        raise RuntimeError("Could not infer a painted worm color. Set PAINT_WORM_RGB manually.")

    candidate_pixels = rgb[candidate_mask]
    quantized = (candidate_pixels // 16) * 16
    colors, counts = np.unique(quantized.reshape(-1, 3), axis=0, return_counts=True)
    saturation = colors.max(axis=1).astype(np.float32) - colors.min(axis=1).astype(np.float32)
    score = counts.astype(np.float32) * (1.0 + saturation / 255.0)
    color_bin = colors[int(np.argmax(score))]
    in_bin = np.all(quantized == color_bin, axis=1)
    return candidate_pixels[in_bin].mean(axis=0).astype(np.float32)

def make_binary_mask(label_path):
    arr = read_label_image(label_path)
    mode = _label_mode_for_array(arr)

    if mode == "value":
        label = arr[..., 0] if _rgb_like(arr) else arr
        if WORM_LABEL_VALUE is None:
            raise RuntimeError("WORM_LABEL_VALUE is not set for value-mode labels.")
        return (label == WORM_LABEL_VALUE).astype(np.uint8)

    if not _rgb_like(arr):
        raise RuntimeError(f"{label_path.name} is not an RGB/RGBA label image. Use LABEL_MASK_MODE='value'.")

    if arr.shape[-1] >= 4 and mode == "paint_color":
        alpha = arr[..., 3]
        if np.any(alpha <= PAINT_ALPHA_THRESHOLD) and np.any(alpha > PAINT_ALPHA_THRESHOLD):
            return (alpha > PAINT_ALPHA_THRESHOLD).astype(np.uint8)

    rgb = _rgb_view(arr)
    if mode == "paint_color":
        worm_rgb = _estimate_worm_rgb(rgb)
        return (_color_distance(rgb, worm_rgb) <= PAINT_COLOR_TOLERANCE).astype(np.uint8)
    if mode == "non_background":
        background_rgb = _estimate_background_rgb(rgb)
        return (_color_distance(rgb, background_rgb) > PAINT_COLOR_TOLERANCE).astype(np.uint8)

    raise ValueError(f"Unknown LABEL_MASK_MODE: {mode}")

def describe_label(label_path):
    arr = read_label_image(label_path)
    mode = _label_mode_for_array(arr)
    if mode == "value":
        label = read_label_array(label_path)
        values, counts = np.unique(label, return_counts=True)
        return f"mode=value values={dict(zip(values.tolist(), counts.tolist()))}"

    rgb = _rgb_view(arr)
    background_rgb = tuple(_estimate_background_rgb(rgb).astype(int).tolist())
    if mode == "paint_color":
        worm_rgb = tuple(_estimate_worm_rgb(rgb).astype(int).tolist())
        return f"mode=paint_color worm_rgb={worm_rgb} background_rgb={background_rgb}"
    return f"mode=non_background background_rgb={background_rgb}"

value_counts = {}
sample_modes = []
sample_label_paths = [label_path for _, label_path in pairs[: min(len(pairs), 25)]]
for label_path in sample_label_paths:
    arr = read_label_image(label_path)
    mode = _label_mode_for_array(arr)
    sample_modes.append(mode)
    print(label_path.name, describe_label(label_path))
    if mode == "value":
        label = read_label_array(label_path)
        values, counts = np.unique(label, return_counts=True)
        for v, c in zip(values.tolist(), counts.tolist()):
            value_counts[int(v)] = value_counts.get(int(v), 0) + int(c)

if WORM_LABEL_VALUE is None and value_counts:
    nonzero_values = sorted([v for v in value_counts if v != 0])
    if not nonzero_values:
        raise RuntimeError("No non-zero label values found. Check that labels were exported correctly.")
    WORM_LABEL_VALUE = nonzero_values[-1]

if value_counts:
    print("\nCombined sampled label values:", value_counts)
if WORM_LABEL_VALUE is not None:
    print("Using WORM_LABEL_VALUE =", WORM_LABEL_VALUE)

coverages = [float(make_binary_mask(label_path).mean()) for label_path in sample_label_paths]
print("Sample worm mask coverage:", f"{np.mean(coverages) * 100:.2f}%", "of pixels")
if np.mean(coverages) > 0.80:
    print("Warning: the selected worm mask covers most of the image. Check label mode, value, or colors.")
elif np.mean(coverages) < 0.001:
    print("Warning: the selected worm mask covers almost none of the image. Check label mode, value, or colors.")

## 5. Visualize matched examples

The middle panel shows the binary mask that will be used for training, and the right panel overlays those pixels in red.

In [ ]:
def load_image_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))

def show_examples(pairs, count=4):
    count = min(count, len(pairs))
    sample_pairs = random.sample(pairs, count)
    fig, axes = plt.subplots(count, 3, figsize=(12, 4 * count))
    if count == 1:
        axes = np.expand_dims(axes, 0)
    for row, (image_path, label_path) in enumerate(sample_pairs):
        image = load_image_rgb(image_path)
        mask = make_binary_mask(label_path)

        overlay = image.copy()
        overlay[mask > 0] = (255, 0, 0)
        blend = cv2.addWeighted(image, 0.60, overlay, 0.40, 0)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(image_path.name)
        axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title(label_path.name)
        axes[row, 2].imshow(blend)
        axes[row, 2].set_title("Training overlay: red = worm")
        for ax in axes[row]:
            ax.axis("off")
    plt.tight_layout()

show_examples(pairs, count=4)

## 6. Train/validation/test split

In [ ]:
random.Random(SEED).shuffle(pairs)

n_total = len(pairs)
n_test = max(1, int(round(n_total * TEST_SPLIT))) if n_total >= 10 else 0
n_val = max(1, int(round(n_total * VALIDATION_SPLIT))) if n_total >= 5 else 0

test_pairs = pairs[:n_test]
val_pairs = pairs[n_test:n_test + n_val]
train_pairs = pairs[n_test + n_val:]

if not train_pairs:
    raise RuntimeError("Not enough matched pairs for training.")

print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

## 7. Build TensorFlow datasets

In [ ]:
def load_pair_np(image_path, label_path):
    image_path = image_path.decode("utf-8")
    label_path = label_path.decode("utf-8")

    image = Image.open(image_path).convert("RGB")
    mask = make_binary_mask(label_path)

    image = image.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.Resampling.BILINEAR)
    mask = Image.fromarray(mask).resize((IMG_SIZE[1], IMG_SIZE[0]), Image.Resampling.NEAREST)

    image = np.asarray(image).astype(np.float32) / 255.0
    mask = np.asarray(mask).astype(np.float32)[..., np.newaxis]
    return image, mask

def load_pair_tf(image_path, label_path):
    image, mask = tf.numpy_function(load_pair_np, [image_path, label_path], [tf.float32, tf.float32])
    image.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    mask.set_shape((IMG_SIZE[0], IMG_SIZE[1], 1))
    return image, mask

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask

def make_dataset(pair_list, training=False):
    image_paths = [str(p[0]) for p in pair_list]
    label_paths = [str(p[1]) for p in pair_list]
    ds = tf.data.Dataset.from_tensor_slices((image_paths, label_paths))
    if training:
        ds = ds.shuffle(buffer_size=len(pair_list), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_pair_tf, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_pairs, training=True)
val_ds = make_dataset(val_pairs, training=False) if val_pairs else None
test_ds = make_dataset(test_pairs, training=False) if test_pairs else None

for batch_images, batch_masks in train_ds.take(1):
    print(batch_images.shape, batch_masks.shape, batch_images.dtype, batch_masks.dtype)
    print("Mask min/max:", tf.reduce_min(batch_masks).numpy(), tf.reduce_max(batch_masks).numpy())

## 8. Define a compact U-Net model

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def build_unet(input_shape=(256, 256, 3)):
    inputs = keras.Input(shape=input_shape)

    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 64)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 128)
    p4 = layers.MaxPooling2D()(c4)

    bn = conv_block(p4, 256)

    u4 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(bn)
    u4 = layers.Concatenate()([u4, c4])
    c5 = conv_block(u4, 128)

    u3 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(c5)
    u3 = layers.Concatenate()([u3, c3])
    c6 = conv_block(u3, 64)

    u2 = layers.Conv2DTranspose(32, 2, strides=2, padding="same")(c6)
    u2 = layers.Concatenate()([u2, c2])
    c7 = conv_block(u2, 32)

    u1 = layers.Conv2DTranspose(16, 2, strides=2, padding="same")(c7)
    u1 = layers.Concatenate()([u1, c1])
    c8 = conv_block(u1, 16)

    outputs = layers.Conv2D(1, 1, activation="sigmoid", name="worm_probability")(c8)
    return keras.Model(inputs, outputs, name="worm_crop_unet")

def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

def binary_iou(y_true, y_pred):
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred > PREDICTION_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + 1.0) / (union + 1.0)

model = build_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=bce_dice_loss,
    metrics=[dice_coefficient, binary_iou, "binary_accuracy"],
)
model.summary()

## 9. Train

In [ ]:
best_model_path = OUTPUT_DIR / "worm_crop_model_best.keras"
last_model_path = OUTPUT_DIR / "worm_crop_model_last.keras"
history_csv_path = OUTPUT_DIR / "training_history.csv"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_dice_coefficient" if val_ds is not None else "dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss" if val_ds is not None else "loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss" if val_ds is not None else "loss",
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(history_csv_path)),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(last_model_path)
print("Saved last model to", last_model_path)
print("Best model path", best_model_path)

## 10. Plot training history

In [ ]:
def plot_history(history):
    metrics = ["loss", "dice_coefficient", "binary_iou"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
    for ax, metric in zip(axes, metrics):
        ax.plot(history.history.get(metric, []), label=metric)
        val_metric = "val_" + metric
        if val_metric in history.history:
            ax.plot(history.history[val_metric], label=val_metric)
        ax.set_xlabel("Epoch")
        ax.set_title(metric)
        ax.grid(True)
        ax.legend()
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "training_history.png"
    plt.savefig(plot_path, dpi=150)
    print("Saved", plot_path)

plot_history(history)

## 11. Evaluate and visualize predictions

In [ ]:
if best_model_path.exists():
    model = keras.models.load_model(
        best_model_path,
        custom_objects={
            "bce_dice_loss": bce_dice_loss,
            "dice_loss": dice_loss,
            "dice_coefficient": dice_coefficient,
            "binary_iou": binary_iou,
        },
    )
    print("Loaded best model")

if test_ds is not None:
    print("Test metrics:")
    print(model.evaluate(test_ds, return_dict=True))
elif val_ds is not None:
    print("Validation metrics:")
    print(model.evaluate(val_ds, return_dict=True))

In [ ]:
def predict_resized_mask(image_path):
    image = Image.open(image_path).convert("RGB")
    resized = image.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.Resampling.BILINEAR)
    arr = np.asarray(resized).astype(np.float32) / 255.0
    pred = model.predict(arr[np.newaxis, ...], verbose=0)[0, ..., 0]
    return np.asarray(resized), pred

def largest_component_box(binary_mask, margin=0):
    mask_uint8 = binary_mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    if num_labels <= 1:
        return None
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    x = int(stats[largest_label, cv2.CC_STAT_LEFT])
    y = int(stats[largest_label, cv2.CC_STAT_TOP])
    w = int(stats[largest_label, cv2.CC_STAT_WIDTH])
    h = int(stats[largest_label, cv2.CC_STAT_HEIGHT])

    x0 = max(0, x - margin)
    y0 = max(0, y - margin)
    x1 = min(binary_mask.shape[1], x + w + margin)
    y1 = min(binary_mask.shape[0], y + h + margin)
    return x0, y0, x1, y1

def show_predictions(pair_list, count=4):
    if not pair_list:
        pair_list = train_pairs
    count = min(count, len(pair_list))
    sample_pairs = random.sample(pair_list, count)
    fig, axes = plt.subplots(count, 4, figsize=(16, 4 * count))
    if count == 1:
        axes = np.expand_dims(axes, 0)

    for row, (image_path, label_path) in enumerate(sample_pairs):
        image_resized, pred = predict_resized_mask(image_path)
        pred_binary = pred >= PREDICTION_THRESHOLD
        true_mask = Image.fromarray(make_binary_mask(label_path)).resize((IMG_SIZE[1], IMG_SIZE[0]), Image.Resampling.NEAREST)
        true_mask = np.asarray(true_mask)

        crop_preview = image_resized.copy()
        box = largest_component_box(pred_binary, margin=CROP_MARGIN_PIXELS_ON_RESIZED_IMAGE)
        if box is not None:
            x0, y0, x1, y1 = box
            cv2.rectangle(crop_preview, (x0, y0), (x1, y1), (255, 0, 0), 2)

        axes[row, 0].imshow(image_resized)
        axes[row, 0].set_title("Image")
        axes[row, 1].imshow(true_mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("True worm mask")
        axes[row, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title("Predicted probability")
        axes[row, 3].imshow(crop_preview)
        axes[row, 3].set_title("Crop box preview")
        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    preview_path = OUTPUT_DIR / "prediction_crop_previews.png"
    plt.savefig(preview_path, dpi=150)
    print("Saved", preview_path)

show_predictions(test_pairs if test_pairs else val_pairs, count=4)

## 12. Export model artifacts

The `.keras` file is useful for continued training. The SavedModel or TFLite outputs are better for deployment conversion. For WormProfiler, a good next step is to convert the SavedModel to ONNX and run it with OpenCV DNN or ONNX Runtime in C++.

In [ ]:
saved_model_dir = OUTPUT_DIR / "worm_crop_saved_model"
tflite_path = OUTPUT_DIR / "worm_crop_model.tflite"
metadata_path = OUTPUT_DIR / "worm_crop_model_metadata.json"

# Keras 3 uses model.export for SavedModel. Fall back for older versions.
try:
    model.export(saved_model_dir)
except AttributeError:
    tf.saved_model.save(model, saved_model_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
tflite_model = converter.convert()
tflite_path.write_bytes(tflite_model)

metadata = {
    "input_height": IMG_SIZE[0],
    "input_width": IMG_SIZE[1],
    "input_channels": 3,
    "input_dtype": "float32",
    "input_scale": "image_rgb / 255.0",
    "output": "single-channel sigmoid worm probability mask",
    "prediction_threshold": PREDICTION_THRESHOLD,
    "label_mask_mode": LABEL_MASK_MODE,
    "worm_label_value_used_for_training": None if WORM_LABEL_VALUE is None else int(WORM_LABEL_VALUE),
    "paint_worm_rgb": None if PAINT_WORM_RGB is None else list(PAINT_WORM_RGB),
    "paint_background_rgb": None if PAINT_BACKGROUND_RGB is None else list(PAINT_BACKGROUND_RGB),
    "paint_color_tolerance": PAINT_COLOR_TOLERANCE,
    "label_suffix": "_Labels",
}
metadata_path.write_text(json.dumps(metadata, indent=2))

print("SavedModel:", saved_model_dir)
print("TFLite:", tflite_path)
print("Metadata:", metadata_path)
print("Best Keras model:", best_model_path)

## Optional: Convert to ONNX

ONNX is often the cleanest route for C++ deployment. Run this cell if you want an `.onnx` model for OpenCV DNN or ONNX Runtime.

In [ ]:
# !pip -q install tf2onnx
# onnx_path = OUTPUT_DIR / "worm_crop_model.onnx"
# !python -m tf2onnx.convert --saved-model "{saved_model_dir}" --output "{onnx_path}" --opset 13
# print("ONNX:", onnx_path)

## Deployment note for WormProfiler

The C++ app should reproduce the same preprocessing used here:

```text
1. Load original image as RGB/BGR.
2. Resize to 256 x 256.
3. Convert to float32 and divide by 255.
4. Run the segmentation model.
5. Threshold output probability mask at 0.5.
6. Keep the largest connected component.
7. Scale the bounding box back to the original image size.
8. Add a margin.
9. Crop the original full-resolution image.
```

If the model predicts no confident component, WormProfiler should skip auto-cropping and continue with the original image or show a warning.